# OCR en Colab — carril sin instalación**Curso Data Science con Python · Universidad del Pacífico · Semana 10***Document AI y voz: OCR, extracción estructurada y ASR*Este notebook corre el **mismo código** que la versión local. Lo único que cambia es lamáquina: acá hay una GPU T4 gratuita.> Esa frase es, en sí misma, la lección de la clase.### Antes de empezarMenú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**.Si no lo haces, todo funciona igual pero mucho más lento, y el `doctor` te lo va adecir en la cara.

## Celda 1 — InstalaciónDos cosas que vas a hacer mal si nadie te las dice:1. **No hay que instalar CUDA.** PaddlePaddle lo gestiona solo; basta el driver NVIDIA,   que en Colab ya está. Quien abra la página de descarga de NVIDIA acaba de perder   media hora.2. **Nunca instalar `paddlepaddle` y `paddlepaddle-gpu` a la vez.** Conviven sin lanzar   error, y el resultado es que todo corre en CPU 40 veces más lento **sin avisar**.Esto tarda unos 3 minutos. No mires la barra: lee la celda siguiente mientras baja.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
!python -m pip install -q paddlepaddle-gpu==3.3.0 \
  -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ \
  --extra-index-url https://pypi.org/simple
!python -m pip install -q "paddleocr==3.7.0" "pypdfium2==4.30.0" "pypdf==5.1.0" \
  "pyyaml==6.0.2"
print("Instalación terminada.")

## Celda 2 — Traer el repositorio y el documentoEl PDF es público: son las bases de la Licitación Pública de Obras N° 05-2026-C-MPI dela Municipalidad Provincial de Ilo, descargadas de SEACE.Si el profesor te dio una URL del repositorio, usa la primera opción. Si no, sube losarchivos a mano con el panel de la izquierda.

In [ ]:
import os
from pathlib import Path

# Opción A: clonar el repositorio del curso (reemplaza la URL)
# !git clone https://github.com/USUARIO/REPO.git ocr_clase
# os.chdir("ocr_clase")

# Opción B: subir a mano src/, config.yaml y el PDF con el panel de archivos,
# y dejarlos con esta estructura:
#   config.yaml
#   src/*.py
#   data/raw/BASE-ESTANDAR-ILO-SEACE-OCR.pdf

for carpeta in ("data/raw", "data/processed", "data/outputs", "logs"):
    Path(carpeta).mkdir(parents=True, exist_ok=True)

print("Carpeta de trabajo:", Path.cwd())
print("Archivos:", sorted(p.name for p in Path.cwd().iterdir()))

## Celda 3 — El doctor: ¿la GPU está **realmente** en uso?PaddlePaddle **no lanza error cuando la GPU falla**: cae a CPU en silencio y siguefuncionando. Crees que tienes aceleración y no la tienes.> Analogía: tienes bicicleta y auto. Le dices al chofer "llévame". Si la llave del auto> no está colgada, no te avisa: te lleva en bicicleta. Llegas tarde y nunca te enteras> de que existía el auto.Seis chequeos. Los cinco primeros pueden dar verde y aun así el modelo correr en CPU.**El sexto es el único que no se puede falsificar:** hace una inferencia de verdad ymira si la VRAM se mueve.

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd()))

from src.comun import cargar_config
from src import doctor

config = cargar_config()
for c in doctor.diagnosticar(config):
    print(c.linea())

## Celda 4 — Triaje: la regla de oro> Usar OCR **solo** en las páginas que son imagen. En el resto, extracción directa, que> es gratis e instantánea.Sobre este documento el resultado es extremo y limpio: **57 páginas, 0 con texto, 57imagen**. Cero por ciento aprovechable sin OCR.**Pregunta:** si de un documento de 500 páginas solo 20 son escaneadas, ¿cuánto teahorras haciendo triaje primero?

In [ ]:
from src import triaje

filas = triaje.ejecutar(config)

## Celda 5 — Preparación: aquí se gana la peleaEste PDF trae tres problemas de golpe:1. Entre 9 y 63 **imágenes embebidas por hoja**: el escáner guardó mosaicos. Hay que   rasterizar la página completa, no extraer sus imágenes.2. **Dos páginas lógicas por hoja**, una sobre la otra. Hay que partir.3. El contenido está **girado**. Hay que enderezar.El orden es: rasterizar → partir → enderezar → OCR. Partir después del OCR no sirve: elmodelo ya mezcló las dos páginas.

In [ ]:
from src import preparar

generadas = preparar.ejecutar(hoja_inicial=1, hoja_final=1, config=config)
for ruta in generadas:
    print(ruta.name)

In [ ]:
from PIL import Image

pagina = Image.open(generadas[0])
print(f"{generadas[0].name}: {pagina.size[0]}x{pagina.size[1]} px")
pagina.resize((pagina.size[0] // 4, pagina.size[1] // 4))

## Celda 6 — El primer OCREl OCR no devuelve texto. Devuelve **texto + geometría + confianza**, y los tres datosvalen igual:| Campo | Para qué sirve ||---|---|| `texto` | lo obvio || `caja` | resaltar en el original lo que la IA afirma || `confianza` | decidir qué revisa un humano |

In [ ]:
from src import ocr_run

motor = ocr_run.construir_ocr(config)
lineas = ocr_run.ocr_imagen(generadas[0], config, motor)

print(f"{len(lineas)} líneas\n")
for linea in lineas[:15]:
    print(f"  {linea.confianza:.3f}  {linea.texto}")

## Celda 7 — El momento GPU: medir sin autoengañarse> Analogía: cronometras cuánto tarda tu mamá en servir la sopa, pero arrancas el> cronómetro cuando todavía está prendiendo la cocina. El primer plato siempre parece> lentísimo.La primera inferencia incluye cargar el modelo a la VRAM e inicializar CUDA. Por eso elprotocolo **descarta páginas de calentamiento**, mide varias, y reporta la **mediana**,no el promedio: un pico aislado no debe mover la conclusión.

In [ ]:
from src import bench

# En Colab reducimos la matriz para que quepa en el tiempo de clase.
config["benchmark"]["paginas_calentamiento"] = 1
config["benchmark"]["paginas_medidas"] = 4

mediciones = bench.ejecutar(config, rapido=True)

### Lee la tabla en este orden1. La **VRAM** sube en modo GPU y queda plana en modo CPU.2. La **mediana de segundos por página** cae de forma drástica.3. El **texto reconocido es el mismo**. Solo cambia el tiempo.Referencia del caso real que da origen a esta clase, sobre una GTX 1660 Ti de 6 GB:~45 s por página en CPU contra ~1 s en GPU. Un rango de 42 páginas pasó de 32 minutos acerca de 1 minuto.**Honestidad obligatoria:** parte de ese 40x es "GPU buena" y parte es "CPU malconfigurada". Un CPU con inferencia optimizada acorta bastante la brecha. Prometer 40xsin matices es vender humo.

## Celda 8 — Provocar el fallback mudoAhora construimos el motor **sin** el parámetro `device`. Todo va a funcionar. Nada vaa avisar. La VRAM no se va a mover. El tiempo se multiplica.> Esto es lo más caro que te puede pasar en un proyecto real: creer que tienes> aceleración y no tenerla. Y fíjate que no hay ningún error en pantalla.> **El silencio es el problema.**

In [ ]:
import time
from paddleocr import PaddleOCR

base = doctor.leer_vram_usada_mib()
print(f"VRAM antes: {base:.0f} MiB")

# Sin device: PaddleOCR decide por su cuenta, y no avisa qué decidió.
motor_mudo = PaddleOCR(lang="es")

t0 = time.perf_counter()
_ = motor_mudo.predict(str(generadas[0]))
print(f"Tiempo: {time.perf_counter() - t0:.2f} s")

pico = doctor.leer_vram_usada_mib()
print(f"VRAM después: {pico:.0f} MiB (delta {pico - base:+.0f} MiB)")
print()
print("Si el delta es pequeño, el modelo está en CPU y nadie te lo dijo.")
print("Por eso ocr_run.py registra en el log el dispositivo REAL que reporta Paddle,")
print("no el que se le pidió. Lo que importa es dónde corrió.")

## Celda 9 — Extracción de los cuatro camposDe texto a datos. Dos vías, y la comparación **es** la lección.**La regla de verificación**, heredada del proyecto real:> Todo dato extraído debe existir **literalmente** en el texto del OCR. Si no existe, se> marca `sin_verificar`.Un modelo que "corrige" el número de licitación y acierta es tan peligroso como uno quelo inventa, porque la próxima vez va a corregir y fallar.

In [ ]:
from src import extraer

# Las dos mitades de la hoja: el nombre de la entidad vive en el encabezado de la
# hoja física, que al partirla cae en la otra página lógica.
ocr_run.ocr_lote(generadas, config)

texto_ocr = extraer.texto_de_hojas(config, hojas=[1])
campos = config["extraccion"]["campos"]

extracciones = [extraer.extraer_por_reglas(texto_ocr, campos)]
por_modelo = extraer.extraer_por_modelo(texto_ocr, campos)
if por_modelo:
    extracciones.append(por_modelo)

extraer.imprimir_comparacion(extracciones, campos)

## Celda 10 — Cierre### El dato que contradice la intuición> En el ranking público de parsing de documentos, los tres primeros puestos de 2026 los> tienen modelos de **menos de 1.3 mil millones** de parámetros. Un modelo de **235 mil> millones** queda por debajo.El pequeño especializado le gana al gigante generalista en su propio terreno.### Y el más útil para licitaciones> En Wild-OmniDocBench, que usa **fotos** de las mismas páginas, el mejor modelo cae de> 91.93 a 72.19. **Veinte puntos** de caída por calidad de imagen.Traducción: invertir en preprocesamiento rinde más que cambiar de modelo.### Tu entregable1. El CSV con los cuatro campos.2. Una línea: ¿en qué modo corriste, y cuántos segundos por página te tomó?La segunda parte es la que importa: te obliga a haber medido.### Sigue con- `notebooks/01_mini_zaragoza.ipynb` — el pipeline completo paso a paso.- `fallas/README.md` — tres fallas sembradas; diagnostica leyendo el log.